# Reverse-complement pks1 chimeras — `reverse-complement-20260829-192804.fasta`

**Self-contained.** The backbone, all 30 FASTA records, and the 21 previously-published chimeras are
embedded below. Only dependency is Biopython (auto-installed if missing). No data files.

---

## Bottom line

Grafting the **reverse-complement** FASTA onto the wild-type `pks1` backbone yields **21 chimeras,
of which 13 are new** and **8 are byte-identical to chimeras you already have**.

The 8 collisions are not a bug. They are a consequence of how the original set was built:

> The original run **already searched both orientations** of every donor and kept the higher-scoring
> one. Eight of the 21 winners were the reverse-complement orientation — those are exactly the eight
> that reappear here. Re-running the *identical* pipeline (strand search included) on the
> reverse-complement file returns the original 21 unchanged, because reverse-complementing the input
> does not change the search space.

So to get anything new, the orientation must be **forced** to the one in the file rather than
searched. `MODE = "as_given"` does that and is the default. `MODE = "best_strand"` reproduces the
original 21 exactly and is used as the self-test.

Net effect: 13 additional distinct 800-nt candidates for the Judge, each the losing-orientation
graft of a donor region whose winning orientation you have already scored. That is a paired design —
`chimeraNN` vs `rcNN` on the same donor region — which is more informative per API call than 13
unrelated candidates.

---

## Method (reverse-engineered from your existing CSV, then verified)

The generating pipeline was not in the files you uploaded, so it was recovered by grid search
against the 21 published sequences. The following recipe reproduces **all 21 byte-for-byte**
(§4 asserts this and halts if it ever stops holding):

| Step | Setting |
|---|---|
| Dedup | reverse-complement-aware, keep first occurrence → 30 records collapse to **21 regions** |
| Aligner | `Bio.Align.PairwiseAligner`, `mode="local"` (Smith–Waterman) |
| Scores | match `+2`, mismatch `−1`, gap open `−5`, gap extend `−2` |
| Alignment chosen | `alignments[0]` (first co-optimal path) |
| Graft rule | at columns where **both** target and donor are ungapped, backbone base ← donor base |
| Donor insertion (gap in backbone) | dropped |
| Donor deletion (gap in donor) | backbone base kept — this is the "keep the letters in between" rule |
| Non-ACGT donor base | **backbone base kept** — this is why the outputs contain zero `N` |

The last row is not cosmetic. `BcssDV1|OR146568.1` carries 24 IUPAC ambiguity codes
(`R Y S W K M N`). Grafting them verbatim would put ambiguity codes into a submission that must be
pure ACGT. Skipping them is what the original run did, and it is the only rule of the four
candidates tested that reproduces `chimera12` exactly.

**Caveat that matters:** several donors have >1000 co-optimal alignments, and different co-optimal
paths give *different* chimeras. `alignments[0]` is a tie-break internal to Biopython, so the exact
output is pinned to the Biopython version. Verified on **1.88**. The §4 self-test is the guard: if a
future version reorders co-optimal paths, it fails loudly instead of silently emitting a different
sequence set.

---
## 1. Configuration

In [1]:
import subprocess, sys

try:
    import Bio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=False)
    import Bio

from Bio import Align
print("biopython", Bio.__version__, "(pipeline verified on 1.88)")

# "as_given"    -> graft the donor in the orientation it appears in the FASTA. No strand search.
#                  This is the one that produces new material from a reverse-complement file.
# "best_strand" -> search both orientations, keep the higher-scoring one. Reproduces the
#                  original 21 chimeras. Used by the self-test in section 4.
MODE = "as_given"

MATCH, MISMATCH, GAP_OPEN, GAP_EXTEND = 2, -1, -5, -2

OUT_CSV      = "pks1_mycovirus_chimeras_revcomp.csv"
OUT_CSV_NEW  = "pks1_mycovirus_chimeras_revcomp_NEW_ONLY.csv"
OUT_DIAG     = "pks1_mycovirus_chimeras_revcomp_diagnostics.csv"
OUT_FASTA    = "pks1_mycovirus_chimeras_revcomp.fasta"

GATUNEK       = "Trichoderma atroviride P1 (mycovirus chimera)"
GATUNEK_KROTKO = "T. atroviride P1"
GENOM         = "pks1_dziki_backbone"

print("MODE =", MODE)

biopython 1.88 (pipeline verified on 1.88)
MODE = as_given


---
## 2. Embedded data

### 2.1 Wild-type backbone (`/dziki`, 800 nt)

In [2]:
DZIKI = (
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGAGGTTGATCGATGGAAATGTATATGTGTATAAG"
    "AGTAGTAAGACTATTGTTTGTAAGATTTACATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC")

assert len(DZIKI) == 800 and set(DZIKI) <= set("ACGT")
print("backbone: %d nt, pure ACGT" % len(DZIKI))

backbone: 800 nt, pure ACGT


### 2.2 The reverse-complement FASTA, verbatim

Embedded exactly as uploaded — 30 records, headers included, so the notebook needs no file.

In [3]:
FASTA_TEXT = r""">SsHADV-1|GQ365709.1|IR1|divergent|AC1 reverse complement
GTGATAAATTGTTGTCCCTTTGAGTCTCTGGGCTAAACTAATATTATAGCCCAGGGACGG
GGACACTTCCAACTATAAATACCCCGTCACTTCCCACGCAACGGACAATCAAACATGTCC
GCATCATCACAAG

>SsHADV-1|GQ365709.1|IR1|divergent|V1 reverse complement
CTTGTGATGATGCGGACATGTTTGATTGTCCGTTGCGTGGGAAGTGACGGGGTATTTATA
GTTGGAAGTGTCCCCGTCCCTGGGCTATAATATTAGTTTAGCCCAGAGACTCAAAGGGAC
AACAATTTATCAC

>FgGMTV1|NC_078279.1|IR0|terminal|CP reverse complement
TTTTGTCGACGTAATAATGTTTTGTGTTAAAAAATAAATTTTTTAAGTCCAAAAACAGAC
TAGCAAAAATGGTGGCTACCCCTCCCTTATATACCCAAAATCAAACCTGGGTCACGGTCA
CAAATCTAAGAAAGAGAAGAA

>FgGMTV1|NC_078278.1|IR0|terminal| reverse complement
AAATAAATTTCCGCTGCCTAAATCTGCCTCATTCTGCCTAATCCTACCCTGCTGTGTTCC
ACCCAATTAAAATTGTGCCGTCCCACAATTCTTCATGTCACCACATTTCCACATGTCACG
TGAAATTGCGGGTAACATTCCAATCCTCCACCAATAACGTCGTCAAGGCCAAAGGGAGTC
ACGGTCAGACTTTTAGGAAAGAGAA

>FgGMTV1|NC_078277.1|IR0|terminal| reverse complement
CTAGTATATAAATCAAACTTGGGTCACGGTCACAAATCTAAGAAAGAGAAGAA

>FgGMTV1|MK430077.2|IR0|terminal|coat protein reverse complement
TTTTGTCGACGTAATAATGTTTTGTGTTAAAAAATAAATTTTTTAAGTCCAAAAACAGAC
TAGCAAAAATGGTGGCTACCCCTCCCTTATATACCCAAAATCAAACCTGGGTCACGGTCA
CAAATCTAAGAAAGAGAAGAA

>FgGMTV1|MK430078.1|IR0|terminal| reverse complement
AAATAAATTTCCGCTGCCTAAATCTGCCTCATTCTGCCTAATCCTACCCTGCTGTGTTCC
ACCCAATTAAAATTGTGCCGTCCCACAATTCTTCATGTCACCACATTTCCACATGTCACG
TGAAATTGCGGGTAACATTCCAATCCTCCACCAATAACGTCGTCAAGGCCAAAGGGAGTC
ACGGTCAGACTTTTAGGAAAGAGAA

>FgGMTV1|MK430076.1|IR0|terminal| reverse complement
CTAGTATATAAATCAAACTTGGGTCACGGTCACAAATCTAAGAAAGAGAAGAA

>BcssDV1|OR146577.1|IR0|tandem_forward|hypothetical protein reverse complement
TAAATAGACTTCGTGAGAAAGCCCTACGTGCTGTGCCGCTCTGCTGTCGTGAAATTATTG
GGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACACCCAAAATAAATTGTGGTCAT
GTCCAAACGCAAATGAAGAGTAAAATTTTAAGACAACAAGTTGTCCTACTCTTCATTTGC
TTGGGCTTTATACCAGTCTGCCTTGCGTGGGGTAGCTTATGTCAGTTCACATGATCGCTA
AATTTCTGGTCGCCTCAGGCTCTCGTTCTGTATCGGGAACTAAAAACGTTGCCCGATCTC
TGACGTGCAACAAAAATAATTACAGCACGTGTACACCCCGTGATACACTATAACAGCCTA
ATCGTTCAGGCTATATCCGCTCGATCCCTCCCTCGTGCCTCGGGGGGAGCCCATCTCGCT
CAAGGACCAAAACTGGTGAACCGGAGGTGGTGTTTATTGTGAATACGCACTCGTTCCCGG
TCGCTCTGCTCCCTCCCAGCGGGGGGGCCCCTACCCCTCCCTTACGTGCTAGCTATAGGC
TATCAACACCCTGTATTTACAGCGCCTTACAGGCTATCACTATAAATTTCACAACATCAC
TATAATTACAGCGCCTTACAGGCTATCACTATAAGTGTCACAACATCACTATAATTACAG
CGCCTTACAGGCTATCACTATAAGTGTCACAACATCACTATAATTACAGCGCCTTACAGG
CTATCACTATAAAAAATAAAACACTCTGTATTTATAGGCTATCCTGAGGATATCGCTAAA
CCTGAATAACAGTCCAGCACAGTCCAACACCTGTATTTATAGGCTATCCTAGCCTACAAA
CATTATGTATCTGATTGAAGTTGTAACCCAACCCAATCAACCTTAGGGAGAACA

>BcssDV1|OR146576.1|IR0|terminal|hypothetical protein reverse complement
ATTGTAATGCGTAATGTAGAATGTAAAATTGAATTTTGTGAATTGATGCTAATGAAAATT
CCAGTGGGAATTTGTCTGCATTAAATAGACTTCGTAAAAAAGCCCTACGTGCTGTGCCGC
TCTGCTGTCGTGAAATTATTGGGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACA
CCCAAAATAAATTGTGGTCATGTCCAAACGCAAATG

>BcssDV1|OR146575.1|IR0|terminal|hypothetical protein reverse complement
TAAATAGACTTCGTGAGAAAGCCCTACGTGCTGTGCCGCTCTGCTGTCGTGAAATTATTG
GGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACACCCAAAATAAATTGTGGTCAT
GTCCAAACGCAAATG

>BcssDV1|OR146574.1|IR0|tandem_forward|hypothetical protein reverse complement
TAAATAGACTTCGTGAGAAAGCCCTACGTGCTGTGCCGCTCTGCTGTCGTGAAATTATTG
GGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACACCCAAAATAAATTGTGGTCAT
GTCCAAACGCAAATGAAGAGTAAAATTTTAAGACAACAAGTTGTCCTACTCTTCATTTGC
TGCTGCCTTAAACCAGTCTGCCTTGCGTGGGGTAGCTTATGTCAGTTCACATGATCGCTA
AATTTCTGGTCGCCTCAGGCTCTCGTTCCGTATCGGGAACTAAAAACATTGCCCTGTCTT
AGATATGCAACAAAAAAAATTACAGCACGTGTACACCCCGTGATACGCTATAAGAGCCTG
ATCATTCAGGCTATATCCGCTCGATCCCTCCCTCGTGCCTCGGGGGGAGCCCATCTCGCT
CAAGGACCAAAACTGGTGAACTGGAAGTGGTGCTTATTGTGAATACGCACTCGTTCCCGG
TCGCGCTGCTCCCTCCCAGCGGGGGGGCCCCTACCCCTCCCTTACGTGCTAGCTATAGGC
TATCAACACCCTGTATTTACAGCGCCTTACAGGCTATCACTATAAATTTCACAACATCAC
TATAATTACAGCGCCTTACAGGCTATCACTATAAATTTCACTACATCACTATAATTACAG
CGCCTTACAGGCTATCACTATAAATTTCACAACATCACTATAATTACAGCGCCTTACAGG
CTATCACTATAAAAAATAAAAACACTCTGTATTTACAGGCTATACAGAGGATATCGCTAA
ACCTGAATAACAGTCCAGCACAGTCCAACACCTGTATTTATAGGCTATCCTAGCCTACAA
ACATTATGTATCTGATTGAAGTTGTAACCCAACCCAATCAACCTTAGGGAGAACA

>BcssDV1|OR146573.1|IR0|terminal|hypothetical protein reverse complement
ATTGTAATGCGTAATGTAGAATGTAAAATTGAATTTTGTGAATTGATGCTAATGAAAATT
CCAGTGGGAATTTGTCTGCATTAAATAGACTTCGTAAAAAAGCCCTACGTGCTGTGCCGC
TCTGCTGTCGTGAAATTATTGGGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACA
CCCAAAATAAATTGTGGTCATGTCCAAACGCAAATG

>BcssDV1|OR146572.1|IR0|terminal|hypothetical protein reverse complement
ATTGTAATGCGTAATGTAGAATGTAAAATTGAATTTTGTGAATTGATGCTAATGAAAATT
CCAGTGGGAATTTGTCTGCATTAAATAGACTTCGTAAAAAAGCCCTACGTGCTGTGCCGC
TCTGCTGTCGTGAAATTATTGGGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACA
CCCAAAATAAAGTGTGGTCATGTCCAAACGCAAATG

>BcssDV1|OR146570.1|IR0|tandem_forward|hypothetical protein reverse complement
ATTGTAATGCGTAATGTGAAATGTAAAATTGAATTTTGTGAATTGATGCTAATGAAAATT
CCAGTGGGAATTTGTCTGCATTAAATAGACTTCGTGAAAAAGCCCTACGTGCTGTGCCGC
TCTGCTGTCGTGAAATTATTGGGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACA
CCCAAAATAAATTGTGGTCATGTCCAAACGCAAATGAAGAGTAAAATTTTAAGACAACAA
GTTGTCCTACTCTTCATTTGCTGCTGCCTTAACGTTTTTTGTGGAATATGCCTAAACGAA
CCTATGTTCACATGATCGCTAAATTTCTGGTCGCCTCAGGCTCCAGCTCCGTATCGGGAA
CTAAAAACATTGCCCTATCTTAGACGTGCAACAAAAAAATTACAGCACGTGCATACCCCG
TGATACGCTATAACTGCCTAATCGTTCAGGCTATATCCGCTCGATCCCTCCCTCGTGCCT
CGGGGGGAGCCCATCTCGCTGAAGGACCAAAAAGGGTGAACCGGAGGTGGTGCTTATTGT
GAATACGCACTCGTTCCCGGTCGCTCTGCTCCCTCCCAGCGGGGGGGCCCCTACCCCTCC
CTTACGTGCTAGCTATAGGCTATCAACACCCTGTAGTTACAGCGCCTTACAGGCTATCAC
TATAAATTTCACTACATCACTATAATTACAGCGCCTTACAGGCTATCACTATAAATCTCA
CAACATCACTATAATTACAGCGCCTTACAGGCTATCACTATAAATGTCACAACATCACTA
TAATTACAGCGCCTTACAGGCTATCACTATAAAAAATAAAAACACTCTGTATTTATAGGC
TATCCTGAGGATATCGCTAAACCTGAATAACAGTCCAGCACAGTCCAACACCTGTATTTA
TAGGCTATCCTAGCCTACAACACTATGTCGCTGATTGAAGTTGTAACCCAACCCAATCAA
CCTTAGGGAGAATA

>BcssDV1|OR146569.1|IR0|terminal|hypothetical protein reverse complement
ATTGTAATGCGTAATGTGAAATGTAAAATTGAATTTTGTGAATTGATGCTAATGAAAATT
CCAGTGGGAATTTGTCTGCATTAAATAGACTTCGTGAAAAAGCCCTACGTGCTGTGCCGC
TCTGCTGTCGTGAAATTATTGGGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACA
CCCAAAATAAATTGTGGTCATGTCCAAACGCAAATG

>BcssDV1|OR146568.1|IR0|tandem_forward|hypothetical protein reverse complement
TAAATAGACTTCRTGARAAAGCCCTACGTGCTGTGCCGCTCTGCTGTCGTGAAATTATTG
GGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACACCCAAAATAAATTGTGGTCAT
GTCCAAACGCAAATGAAGAGTAAAATTTTAAGACAACAAGTTGTCCTACTCTTCATTTGC
TTGGGCTTTATACCAGTCTGCCTTGCGTGGGGTAGCTTATGTCAATTCACATGATCGCTA
AATTTCTGGTCGCCTCAGGCTCTCGTTCCGTATCGGGAACTAAAAACRTTGYYCKMYCTY
AGACGTGCAACNNAAAAATNAATTACAGCACGTGTAYACCCCGTGATACACTATAACAGC
CTAATCGTTCAGGCTATATCCGCTCGATCCCTCCCTCGTGCCTCGGGGGGAGCCCATCTC
GCTCAAGGACCAAAACGGGTGAACYGGARGTGGTGCTTATTGTGAATACGCACTCGTTCC
CGGTCGCTSTGCTCCCTCCCAGCGGGGGGGCCCCTACCCCTCCCTTACGTGCTAGCTATA
GGCTATCAACACCCTRTAWTTACAGCGCCTTACAGGCTATCACTATAAATTYCAMAACAT
CACTATAATTACAGCGCCTTACAGGCTATCACTATAARTGTCACAACATCACTATAATTA
CAGCGCCTTACAGGCTATCACTATAARTGTCACAACATCACTATAWTTACAGCGCCTTAC
AGGCTATCACTATAAAAAATAAAAACACYCTGTATTTATAGGCTATCCTGAGGATATCGC
TAAACCTGAATAACAGTCCAGCACAGTCCAACACCTGTATTTATAGGCTATCCTAGCCTA
CAAACATTATGTATCTGATTGAAGTTGTAACCCAACCCAATCAACCTTAGGGAGAACA

>BcssDV1|OR146567.1|IR0|terminal|hypothetical protein reverse complement
TAAATAGACTTCGTGAGAGAGCCCTACGTGCTGTGCCGCTTCGCTGTCGTGAAATTATTG
GGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACACCCAAAATAAATTGTGGTCAT
GTCCAAACGCAAATG

>BcssDV1|OR146566.1|IR0|terminal|hypothetical protein reverse complement
TAAATAGACTTCGTGAGAGAGCCCTACGTGCTGTGCCGCTTCGCTGTCGTGAAATTATTG
GGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACACCCAAAATAAATTGTGGTCAT
GTCCAAACGCAAATG

>BcssDV1|OR146565.1|IR0|tandem_forward|hypothetical protein reverse complement
ATTGTAATGCGTAATGTAGAATGTAAAATTGAATTTTGTGAATTGATGCTAATGAAAATT
CCAGTGGGAATTTGTCTGCATTAAATAGACTTCGTAAAAAAGCCCTACGTGCTGTGCCGC
TCTGCTGTCGTGAAATTATTGGGTGTGGTCAAACCAACAAATTTCAACCTTGTGACCACA
CCCAAAATAAATTGTGGTCATGTCCAAACGCAAATGAAGAGTAAAATTTTAAGTGAACCA
GTGGTGGTCGGTTGTCGAGCGGGGTAAGAGGCTCTAGGCGTACTGAGTGAAATTGCCCCG
AGGGCAAGTCGTGTTGGTAAAAGGATGACTGGCGCTGGAGCGACGACTGAGGTCGACGAT
CAACGCGAGGACGACCGCTATCACCACGGCGAGTTGTACTATGAGTGCGATAGTTATCCA
GTCCATTCTGGACCGGAAGCCTTCGAAGTTCAAAATCTTGCAAATTCGCAAGACAGGCCT
CTTCAAAGTCTCTCGAAACATCGAACTCTGAGTCGGACATTGGAAATTCCAGTTTGGACT
TGAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCAAATTGGAGAGAGGAGTTCCT
CCATTATATACACAATGTACGCGCCCACCATGCACTGTGCCGCAAAGGTGTAAACACATG
CGGCCCAGGTAACGTGATTAAATCACGTGGGTGGGGTAATTCAAAAGTGAGGCATGGCCA
AACGCTTTCTAAGAGTAAAATTTTAAGCTATAATTACAGCGCCTTACAGGCTATCACTAT
AAATTGTCACAACATCCCTATATTTACAGCGCCTTACAGGCTATCACTATAAAAAATAAA
AACACTCTGTATTTATAGGCTATCCTGAGGATATCGCTAAACCTGAATAACAGTCCAGCA
CAGTCCAACACCTGTATTTATAGGCTATCCTAGCCTACAACACTATGTCGCTGATTGAAG
TTGTAACCCAACCCAATCAACCTTAGGGAGAATA

>BcssDV1|OR146564.1|IR0|terminal|hypothetical protein reverse complement
TGGAGGTTGCGGTTTGGACTTTAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCA
AATTGGAGAGGGAGGTTCCTCCATTATATACACAATGTACGCGCCCAGTATGCACTGTGC
CGCAAAGGTCTACCATCATGCGACCCCGGTAACGTGATCAAGTCACGTGGGTGGGGTAAT
TGAAAAGTGAGGCATGGCCAAACGCTTTCT

>BcssDV1|OR146563.1|IR0|terminal|hypothetical protein reverse complement
TGAAAATTGCGGTTTGGCCTTGAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCA
AATTGGAGAGAGGAGTTCCTCCATTATATACACAATGTACGCGCCCACCATGCACTGTGC
CGCAAAGGTGTAAACACATG

>BcssDV1|OR146561.1|IR0|terminal|hypothetical protein reverse complement
TGAAAATTGCGGTTTGGCCTTGAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCA
AATTGGAGAGAGGAATTCCTCCAT

>BcssDV1|OR146560.1|IR0|terminal|hypothetical protein reverse complement
TTTGGACTTGAGAGGATAGGTGAAAATG

>BcssDV1|OR146559.1|IR0|terminal|hypothetical protein reverse complement
TGAAAATTGCGGTTTGGCCTTGAGAGGATA

>BcssDV1|OR146558.1|IR0|terminal|hypothetical protein reverse complement
TTTGGACTTGAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCAAATTGGAGAGAG
GAGTTCCTCCATTATATACACAATGTACGCGCCCACCATGCACTGTGCCGCAAAGGTGTA
AACACATGCGGCCCAGGTAACGTGATTAAATCACGTGGGTGGGGTAATTCAAAAGTGAGG
CATGGCCAAACGCTTTCT

>BcssDV1|OR146557.1|IR0|terminal|hypothetical protein reverse complement
TTTGGACTTGAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCAAATTGGAGAGAG
GAGTTCCTCCATTATATACACAATGTACGCGCCCACCATGCACTGTGCCGCAAAGGTGTA
AACACATGCGGCCCAGGTAACGTGATTAAATCACGTGGGTGGGGTAATTCAAAAGTGAGG
CATGGCCAAACGCTTTCT

>BcssDV1|OR146556.1|IR0|terminal|hypothetical protein reverse complement
TTTGGACTTGAGAGGATAGGTGAAAATGTAAAAGACAATAGAAACTCAAATTGGAGAGAG
GAGTTCCTCCATTATATACACAATGTACGCGCCCACCATGCACTGTGCCGCAAAGGTGTA
AACACATGCGGCCCAGGTAACGTGATTAAATCACGTGGGTGGGGTAATTCAAAAGTGAGG
CATGGCCAAACGCTTTCT

>DsCDV1|PP056609.1|IR1|divergent|putative coat protein reverse complement
GATCTAGTATCGTCGATAAGACCGACGACGGGTAAAGCCAGTGAAACGCACTGGACCGTA
GTTTGATTGATTGCGTCGAAAACCAGAACGATTATACGAACGACGATAGTATGGACGACG
ATAGGGAGTACGAGGGCGGTAGCCAGAGGAGCGGAATCTACCATACATGATGAGAGAAGA
AGACCAAGTAAGGACAAGTCACCAACTACATGCTTTATGTAGAAGAACAAGAACGCCAAC
AGAACGCGAACGCGGACCACCGCGTTACTTATGGACGCGAGAAAGTAAACAACATTGACA
TGAGGATCAGCGCGGGACACCGGCGCTAAGACGGGAACGGGTGTGTGGTGGGTCACATCA
CGGGCCGCATATGCAGGACGGGAGCCAACAACGAACGACAGCCACTCAACAGCAAACACC
GTGCGGCACAGCAATCCCCGAGTCAGGCAAACACGACCAACGCCGAAACCTCGCCAATCA
CGACGCACCAGACCAGCCACTCAATATGCAAGGACACCAGGACACCACACACCACGGCTG
TAACATTAAAAGTTCGCCGTGGTGTCCTACCCGCTGCACGCAAAACCATTCATCTATATA
AGGGACGCCAGGACACCGGCTAAAATCTGACTGCAGTGTCCTACACACCATCCCAAAGTG
CGGTGAAACCAAGCGCCCACC

>DsCDV1|PP056609.1|IR1|divergent|putative rep protein reverse complement
GGTGGGCGCTTGGTTTCACCGCACTTTGGGATGGTGTGTAGGACACTGCAGTCAGATTTT
AGCCGGTGTCCTGGCGTCCCTTATATAGATGAATGGTTTTGCGTGCAGCGGGTAGGACAC
CACGGCGAACTTTTAATGTTACAGCCGTGGTGTGTGGTGTCCTGGTGTCCTTGCATATTG
AGTGGCTGGTCTGGTGCGTCGTGATTGGCGAGGTTTCGGCGTTGGTCGTGTTTGCCTGAC
TCGGGGATTGCTGTGCCGCACGGTGTTTGCTGTTGAGTGGCTGTCGTTCGTTGTTGGCTC
CCGTCCTGCATATGCGGCCCGTGATGTGACCCACCACACACCCGTTCCCGTCTTAGCGCC
GGTGTCCCGCGCTGATCCTCATGTCAATGTTGTTTACTTTCTCGCGTCCATAAGTAACGC
GGTGGTCCGCGTTCGCGTTCTGTTGGCGTTCTTGTTCTTCTACATAAAGCATGTAGTTGG
TGACTTGTCCTTACTTGGTCTTCTTCTCTCATCATGTATGGTAGATTCCGCTCCTCTGGC
TACCGCCCTCGTACTCCCTATCGTCGTCCATACTATCGTCGTTCGTATAATCGTTCTGGT
TTTCGACGCAATCAATCAAACTACGGTCCAGTGCGTTTCACTGGCTTTACCCGTCGTCGG
TCTTATCGACGATACTAGATC
"""

print("%d records embedded, %d characters" % (FASTA_TEXT.count(">"), len(FASTA_TEXT)))

30 records embedded, 11725 characters


### 2.3 The 21 published chimeras

Not inputs. They are the ground truth for the self-test in §4 and the duplicate check in §6.

In [4]:
PUBLISHED = {
  "chimera01_SsHADV-1_GQ365709.1_+_123bp_58chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGAGGTTGATCGATGGAAATGTATATGTGTATAAG"
    "AGTAGTAAGACTATTGTTTGTAAGATTTACATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGCGGACATGTTTGATTGTCCGTTGCGTGGGAAAGGTGACGGGGTATCTTATAGTTGGAAGTGTCCCCGTCCTGGG"
    "CTATAATATAGTTTAGCCCAGAGACTCAGGCAAGGGACAACAATTTATCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera02_FgGMTV1_NC_078279.1_+_121bp_54chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTTT"
    "CTTAGATTTGTGACCGTGACCCAGCGTTTGATTTTGGGTATATAAGTGGAGGGGTAGCCATCCATTTTTGCTAGTC"
    "TGTTTTTGGACTAAAAAATTTATTTTTTACAAAAACATTATTAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera03_FgGMTV1_NC_078278.1_-_124bp_56chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGTCCCACAATTCTTTGTCACCCATTTCCACATGTCACGTGAAATTGCGGGTAACAT"
    "TCCAATCCTCCACCAATAACGTCGTCTAAGGCCAAAGGGAGTCACGGTCGAGACTTTTAGGAAAGAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGAGGTTGATCGATGGAAATGTATATGTGTATAAG"
    "AGTAGTAAGACTATTGTTTGTAAGATTTACATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera04_FgGMTV1_NC_078277.1_-_36bp_11chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGAGGTTGATCGATGGAAATGTATATGTGTATAAG"
    "AGTAGTAAGACTATTGTTTGTAAGATTTACATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACTTGGGTCACGTGTCAACAAATCTAAGAAAGAGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera05_BcssDV1_OR146577.1_+_754bp_386chg":
    "TATCTGGTTCTCCCTAAGGTTGATTGCGGTTGGACGTTACAATTCAATCAGATACAAATGTTTGTAGGCTAGGATA"
    "GCCTATAAATACATGGACGTGCTGGACTATTCAGGTTTAGCGATTCCTCAGGATACCTATATAATACAGAGTGTTT"
    "TATTTTTTATAGTGATAGCTGAAGGCGCTGTAATTATAGTGATGTTGTCGACACTTATAGTGACTAGCCTGTAAGG"
    "CGCTGTAATTATAGTGATGTTGTGACACTTAGGTAGTGATAGCCTGTAAGGCGCTGTAATTATAGTGATGTATGTG"
    "AAATTTATAGTGATAGCCTGTAAGCTGTAAATACAGGGTGTTGATAGCCTATAGCGTAGCACGTGGGAGGGGTAGG"
    "GGCCCCCCCGCTGGGAGGGAGCAAGCGACCGGGAACGAGTGCGTATTCACATAAACACCACCTCCGGTTCACCAGT"
    "TTTGGTCCTTGAGCGAGATGGGCTCCCCCCGAGGCACGAGGGATCAGGGATCGAAGCGGAATAGCCGAACGATTAG"
    "GCTGTTATAGTGTATCACGGAGGTGTACACGTGCTGTAATTATTTTTGTTGCACGTCAGAGTCGGGCAACGTTTTT"
    "AGTTCACAGAAGAGAGCCTGAGGCGACCAGAAAAGCGATCATGTGAACTGACATAGCTACCCCACGCAAGGCAGAC"
    "TGGTATAAAGCCCAAGCAAATGAAGGTAGGCAACTTGTTGCTTCTTAAAATTTTACTCTTCAATTTGCGTTTGGAC"
    "ATGACCACAATTTATTTTCTTTTACAGAATCATCACCGCC",
  "chimera06_BcssDV1_OR146576.1_+_188bp_82chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGCGTTTGGACGACCACAATTTATATTTGGGTGTG"
    "AGTCACAAGGTTGAAATTTGTTGGTTTGACACACACCCAATAAACATTTCACGACAGCAGAGCGGCAAGCACGTAG"
    "GGCTTTTTTACGAAGTCTATTTAATGCAGATTCCAAATTCCCTTACTGGAATTTATGTCATTAGCATCAATTCAAA"
    "AATTCAATTTTACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera07_BcssDV1_OR146575.1_+_131bp_62chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGCGTTTGGACGACCACAATTTATATTTGGGTGTG"
    "AGTCACAAGGTTGAAATTTGTTGGTTTGACACACACCCAATAATTTCACGACAGCAGAGCGGCACAGCACGTAGGG"
    "CTTTCTTCCACGAAGTCTATTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera08_BcssDV1_OR146574.1_+_749bp_375chg":
    "TATCTGGTTCTCCCTAAGGTTGATTGCGGTTGGACGTTACAATTCAATCAGGGCCATACATAATGTTTGTAGGCAA"
    "TCTTAGGATAGCCTATAAACGCCCTACAGGTGTTGGACTGTGCTGGACTGTTATTCAGGTTTAGCGATATCCTCTG"
    "TATAGCCTGTAAATACAGCAGTGTTTTTATTTTTTATAGTGATAGCCTGTAAGCGCGCTGTAATTATAGTGTATGT"
    "TGCTGAAATTCTATAGTTAGATAGCCTGTAAGGCGCTGTAATTATAGTGATGTAGTGAAATTTATATGTGTATAAG"
    "CCTAGTAAGGCGCTTAATTATAGTGATGTTGTGAAATTTATAGTGATTAGCCAATGTAAGCGCTGTAAATACAGGG"
    "TGTTGATCCTATACTAGCACGTAAGGGAGGGGTAGGGGCCCCCCCTTGGCTGGGAGGGAGCGCGCGACCGGGAACG"
    "AGTGCGTATTCACAATAAGCCACTTCCAGTTCACCAGTTTTGGTCCTTGAGCGAAGATGGGCTCCCCCCGAGGCAC"
    "GACGGGAGAGGATCGAGCGGATATAGCCTGAATGATCAGGCTCTTAGCTAGCGTATCACGGGGTGTACACGTGCTG"
    "TAATTTTTTTTGTTGCATATCTAAGACAGGGCAATGTTTTTAGTTCCCATACGGAACGAAGAGCCTGAGTGCGACC"
    "AGAAATTGCGATCATGTGAACTGACATAAGCTACCCCACGCAAACTGGTTTAAGGCGCAGCAAATGAAGAGTAGGC"
    "ACAACTTGTTGTCTTAAAATTTTACATCTTCATTTGCGCC",
  "chimera09_BcssDV1_OR146572.1_+_188bp_83chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGCGTTTGGACGACCACACTTTATATTTGGGTGTG"
    "AGTCACAAGGTTGAAATTTGTTGGTTTGACACACACCCAATAAACATTTCACGACAGCAGAGCGGCAAGCACGTAG"
    "GGCTTTTTTACGAAGTCTATTTAATGCAGATTCCAAATTCCCTTACTGGAATTTATGTCATTAGCATCAATTCAAA"
    "AATTCAATTTTACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera10_BcssDV1_OR146570.1_+_670bp_327chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACTCCCTAAGGTTGATTGGGTTGGGTTACAACTTCAATCAGCATAGTGTTGTAGGCGG"
    "TAGGATAGCCTATAAATACAGCCGTGTTGGACTGTGCTGGACTGTTATTCAGGTTTAGCGATATCCTCACGGATAG"
    "CCTATAAATACGAGTGTTTTTATTTTTTATAGGATAGCCTGTAAGGTGTAATTATAGTGATGTATGTGACATTTAT"
    "AGTAGATAGCCTGTAAGGCGCTGTAATTATAGTGATGTTGTAGAGATTCTCTATAGTGATAGCTGTAAGGCGCTGC"
    "TAATTATAGTGATGTACTGTGAAATTTATAGTCCAGATAGCCTGTAAGGCTTTGCTGTAACTACAGGGTGTTGATG"
    "CCTACTAGCTAGCACGTAAGGGAGGGGTAGGGGCCTCCCCCGCTGGGAGGGAGCAGAGCGACCGGGAACGAGTGCG"
    "TACACAATAAGCACCACCTCCGGTTCACCCTTTTTGGTCCTTCGTAGCGAGATGGGCCCCCCCGAGGCACGAGGGA"
    "GGATCAGCGGATATAGCCTGCAACGATTAGGCAAGGTTATAGCGTATCACGGGGTATGCACGTGCTGAGTAATTTT"
    "TTTGTTGCACGTCTAAGATAGCAATGTTTTTATGTTCCCGATACGGAGCTGGAGCCGAGGCGACCAGAAATTTAGC"
    "ATCATGTGAACATAGGTTCGTTTACATATTCCACACCGCC",
  "chimera11_BcssDV1_OR146569.1_+_208bp_96chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGCGTTTGGACGACCACAATTTATATTTGGGTGTG"
    "AGTCACAAGGTTGAAATTTGTTGGTTTGACACACACCCAATAAACATTTCACGACAGCAGAGCGGCAAGCACGTAG"
    "GGCTTTTTCACGAAGTCTATTTAATGCAGATTCCAAATTCCCTACTGGAATTTTCATTAGCTCAATTCACAAAATT"
    "CATTTTACATTTCACATTACGCATTACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera12_BcssDV1_OR146568.1_+_682bp_339chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACTCCCTAAGGTTGATTGGGTTGGGTTACAACTTCAATCAGATACATAATGTTTGTAG"
    "GCTAGGATAGCCTATAAATACAGGTGTTGGACTGTGCTGGACTGTTATTCAGGTTTAGCGATATCCTCACGGATAG"
    "CCTATAAATACGAGTGTTTTTATTTTTTATAGGATAGCCTGTAAGGTGTAACTATAGTGATGTATGTGACAATTAT"
    "AGTAGATAGCCTGTAAGGCGCTGTAATTATAGTGATGTTGTAGACATTCTCTATAGTGATAGCTGTAAGGGGACGC"
    "TGTAATTATAGTGATGTTTTGTAATTTATAGTCCAGATAGCCTGTAAGGCTTTGCTGTAACTAGAGGGTGTTGATG"
    "CCTATAGCTAGCACGTAAGGGAGGGGTAGGGGCCCCCCCGCTGGGAGGGAGCAAAGCCGACCGGCGAACGAGTGCG"
    "TACACAATAAGCACCACCTCCGGTTCACCCGTTTTGGTCCTTCGGAGCGAGATGGGCCCCCCCGAGGCACGAGGGA"
    "GGATCAGCGGATATAGCCTGCAACGATTAGGAACTGTTATAGTGTATCACGGGTATACACGGCTGTAATTGATTTT"
    "TTAGTTGCACGTCTAGTATGTACAAGGTTTTTAGTTCCCGATACGGAACGAGAGCCGAGGCGACCAGAAATTTAGC"
    "ATCATGTGAATTGACATAAGCTACCCCACGCAAGGCAGAC",
  "chimera13_BcssDV1_OR146567.1_+_131bp_61chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTCGGTTTTGCGTTTGGACGACCACAATTTATATTTGGGTGTG"
    "AGTCACAAGGTTGAAATTTGTTGGTTTGACACACACCCAATAATTTCACGACAGCGTAAGCGGCACAAGCACGTAG"
    "GGCTCTCTCACGAAGTCTATTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera14_BcssDV1_OR146565.1_+_770bp_381chg":
    "TATCAGGTGTTGGACTGTGCGGACTGTTATTGCAGGTTGTAGCGATATCCTCAGGATAGCCTATAAATACAGAGTG"
    "TTTTTATTTTTTATAGTGATAGCCTGTAAGGGCTTAAATATAGGGATGTTGTGACAATTTATAGTGATGCCTGAAG"
    "GCGCTGTAATTATAGCTGCTAAAATTTTACTCTTAGAAAGGTTTGGCTCATGCCTCACATTTTGAATTACGCCCAC"
    "CCACGTGATTTAAACGTTACTGGGCCCATGTGTTTCACCTTTGCGGGTGCATGGTGGGCGCGTACATGTGTATAAA"
    "TGGAGGAATCCTCTCTCCAATTTGAGTTTCTATTGTCTTTTACATTTTCACCATCCTCTCAAGTCCAATGGAATCA"
    "ATGTCCGACTCGAGTTCATGTTTCGAGAGATTTGAAGAGGCCTTCTTGGAATTTGCAAGATTTTGAACTTCGAAGG"
    "CTTCCAAGGTAACCAGAATGGACTGACTATCGCACTCATAGTACAACTCGCCGTGGGATAGCGGTCGTCCTCGCGT"
    "TGATCGTCGACCTCAGTCGTCGCTCCAGCGCCAGTGCATCCTTTTACCAAGCACGACATTGCCCTCGGGGCATTGA"
    "TGTTCACTCAGTACGCCTAGAGCCTCTTACCCCGCTCGACAACCGACCACCACTAGGGTTCACTTAAAATTTTACT"
    "CTTCATTATGGTCGTTTGGACATGACCACAATTTATTTTGGGTGTGGTCACAAGTCGTTGAAATTTGTTGGTTTGA"
    "CCACACCCAATAATTTCACGACAGCAGAAGCGGCACAGCC",
  "chimera15_BcssDV1_OR146564.1_-_193bp_91chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATGGAGGTTCGGTTTGGACTTTAGAGGATGGAGGTGAAAATGTAAAAGAC"
    "AATAGAAACTCAAATTGGAGAGGGAGGTTCCTCCATTATATACACAATGTACAGCGCCCAGTATGCACTGTGCCGC"
    "AGGTCTACCATCACGACCCCGGTAACGTGATTCCAAGTCACGTGGGTGGGGTAATTGAAAAGTGAGGCATGGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera16_BcssDV1_OR146563.1_-_120bp_54chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGCGGTTTGGCCTTGAGGAGGATAGGTGAAAATGTAAAAGACAATAAA"
    "ACTAATTGGAGAGAGGAGTTCCTCCATTATATACACAATGTACGCGCCCACCATGCACTGTGCCGCAAAGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera17_BcssDV1_OR146561.1_-_79bp_31chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGCGGTTTGGCCTTGAGGAGGATAGGTGAAAATGTAAAAGACAATAAG"
    "AGAACTCAAATTGGAGAGAGGAATTCCTCCATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera18_BcssDV1_OR146560.1_-_28bp_8chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGAGGTTTGGACTTGAGGAGGATAGGTGAAAATGTATATGTGTATAAG"
    "AGTAGTAAGACTATTGTTTGTAAGATTTACATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera19_BcssDV1_OR146559.1_-_19bp_4chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAACAAGGTTGGGCTCGAGGCTTACAGAAACAAGAACGTCTAT"
    "CGCCGAGATTCGAGCGTTACAGAGCCATTGCGGTTTGGCCTTGAGGTTGATCGATGGAAATGTATATGTGTATAAG"
    "AGTAGTAAGACTATTGTTTGTAAGATTTACATAAACATTATAAACATTCTCCAACGTAACAGGTGCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera20_BcssDV1_OR146558.1_+_182bp_79chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTATAGTGCTGTTTGCGG"
    "TCTTTTCCACCACCCCTGCTGCCGCCTCCCACCTAAAAGCGTTTGGCTCATGCCTCACATTTTGAATTACGCCCAC"
    "CCACGTGATTTAAACGTTACTGGGCCCATGTGTTTCACCTTTGCGGGTGCATGGTGGGCGCGTACATGTGTATAAA"
    "TGGAGGAATCCTCTCTCCAATTTGAGTTTCTATTGTCTTTTACATTTTCACCATCCTCTCAAGTCCAAGGGGATGC"
    "TTTTCTTCCTCCACTACTTTTTAATCGAGATTCCAAAACCCCTTCTTGGCTTTTATGTAACTAGCGGCTCTGCATG"
    "CATACAAAGTAACCCAAAGGGACTGACAAGGGGCCTCGGCCGATCAGGGGTCGAAGCCGAACGGCCGCCGACTCAC"
    "TACATGAGAAGGGCCAACGGAGATGCAGCGGAGGTGCAACGTCGTAGCAAGCTCGTCATCGTCGACGAAGGATTGA"
    "TGATCACCGAAGATACTTTGCGTCACGTAGGAAAGGTGTCAAGATATCACAGATAGCGAAGACTCTGAGTGATGCC"
    "CTAGATTATGGTCTTGTATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGC"
    "ACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC",
  "chimera21_DsCDV1_PP056609.1_-_622bp_297chg":
    "TATCTGGTGTTTCATAGCGCGGCCTGCCGTTGGACGTTGCAATGCAAACGGGGCCAGAGATCAAAACTATAGGCAA"
    "TCTTACGATTTCATACACACGCCCAACAAAGGATTCTCTGCCCTTACTACGTGTACCTGTCTAGTATCGTTTGCGA"
    "TAAGACCGACGACGGGTAAAGCCGAAACGCACTGGACCGTAGTTTGATTGATTGCGTCGAAAACCAGAACGATTAT"
    "CGACGAACGACGATAGTTGGACGACGATAGGGGTACGGGGCGGTAGCCAGAGGAGCGGAATCTATACATGTATGAG"
    "AGAAGTAAGACCATAGTAAGGACAAGTCACAACTACATGCTTTATGTAGAAGAACGTAAGAACGCCAACAGAACGC"
    "ACGCGGACCACCGCTACTTTGGACGCGAGAAAGTAAACAACATTCATGAGGATCACGGGACTACCGGCGCTGAAGA"
    "CGGGAACGGGTGTGTGGTGGGTCACACACGGGGCATATGCAGGACGGGAGCCAACAACGAACGACAGCCGACTCAC"
    "ACCGTGGGCACAGCAATCCCCGAGTCAGGCAAACACGACCAACGCCGAAACCTCGCCAATCACGACGCACCAGACC"
    "AGATATGCAAGGACACCAGGCACCACACACCACGGCTGTCAACATATAAAAGTTCGCGTGGTGTCTACCCGCTGCA"
    "CGCAAAACCATTCATCTATATAAGGGACGCCAGGCACCGGCTATCTGACTGCAGTCGGTCCTACACACCATCCCAA"
    "ATGCGGTGAAACCATTTTCTTTTACAGAATCATCACCGCC",
}

assert len(PUBLISHED) == 21 and all(len(s) == 800 for s in PUBLISHED.values())
print("%d published chimeras embedded, all 800 nt" % len(PUBLISHED))

21 published chimeras embedded, all 800 nt


---
## 3. Library

`graft_alignment` is the whole method. Everything else is bookkeeping.

The three cases it distinguishes, in backbone coordinates:

```
backbone  A C G T T A C G      column both ungapped -> substitute
donor     A C - T G A C G      donor gap            -> keep backbone base (T at col 3)
                               donor insertion      -> dropped (never lengthens output)
```

Output length is therefore always exactly 800, unconditionally — asserted in §6.

In [5]:
from collections import OrderedDict

_COMP = str.maketrans("ACGTNacgtnRYSWKMBDHVryswkmbdhv",
                      "TGCANtgcanYRSWMKVHDByrswmkvhdb")


def revcomp(s):
    return s.translate(_COMP)[::-1]


def parse_fasta(text):
    """Returns [(header, sequence), ...] preserving file order."""
    out, name, buf = [], None, []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if name is not None:
                out.append((name, "".join(buf)))
            name, buf = line[1:].strip(), []
        else:
            buf.append(line)
    if name is not None:
        out.append((name, "".join(buf)))
    return [(h, s.upper()) for h, s in out]


def dedup_revcomp_aware(records):
    """Collapse records that are identical OR reverse-complements of each other,
    keeping the first occurrence.

    Reverse-complement-aware is correct even in as_given mode: if X and revcomp(X)
    are both present, forcing forward on the second is the same as forcing reverse
    on the first, which is the orientation the original run already grafted. Keeping
    both would only re-emit an existing chimera under a new name.
    """
    seen, keep = set(), []
    for h, s in records:
        key = min(s, revcomp(s))
        if key in seen:
            continue
        seen.add(key)
        keep.append((h, s))
    return keep


def make_aligner():
    a = Align.PairwiseAligner()
    a.mode = "local"
    a.match_score = MATCH
    a.mismatch_score = MISMATCH
    a.open_gap_score = GAP_OPEN
    a.extend_gap_score = GAP_EXTEND
    return a


def graft_alignment(backbone, alignment):
    """Substitute donor bases into the backbone across the aligned block.

    Returns (sequence, n_changed, n_aligned_columns, n_ambiguous_skipped,
             target_start, target_end).
    """
    out = list(backbone)
    ti = int(alignment.coordinates[0][0])
    n_chg = n_aln = n_skip = 0
    for tc, qc in zip(alignment[0], alignment[1]):
        if tc != "-" and qc != "-":
            n_aln += 1
            if qc in "ACGT":
                if out[ti] != qc:
                    n_chg += 1
                out[ti] = qc
            else:
                n_skip += 1              # IUPAC ambiguity: keep the backbone base
            ti += 1
        elif tc != "-":
            ti += 1                      # gap in donor: keep the backbone base
        # else: gap in backbone (donor insertion) -> drop, never lengthens output
    return ("".join(out), n_chg, n_aln, n_skip,
            int(alignment.coordinates[0][0]), int(alignment.coordinates[0][-1]))


def build_chimera(backbone, donor, aligner, mode):
    """mode='as_given'    -> align the donor as it appears, no strand search
       mode='best_strand' -> align both orientations, keep the higher score (ties -> forward)"""
    if mode == "as_given":
        cands = [("fwd", donor)]
    elif mode == "best_strand":
        cands = [("fwd", donor), ("rev", revcomp(donor))]
    else:
        raise ValueError("mode must be 'as_given' or 'best_strand'")

    best = None
    for orient, q in cands:
        alns = aligner.align(backbone, q)
        if best is None or alns.score > best[0]:
            best = (alns.score, orient, alns)
    score, orient, alns = best
    n_opt = len(alns) if len(alns) < 10 ** 7 else None
    seq, n_chg, n_aln, n_skip, ts, te = graft_alignment(backbone, alns[0])
    return dict(sequence=seq, score=score, orientation=orient, n_optimal=n_opt,
                n_changed=n_chg, n_aligned=n_aln, n_ambiguous_skipped=n_skip,
                target_start=ts, target_end=te, donor_len=len(donor))


RECORDS = parse_fasta(FASTA_TEXT)
REGIONS = dedup_revcomp_aware(RECORDS)
print("%d FASTA records -> %d unique regions after revcomp-aware dedup"
      % (len(RECORDS), len(REGIONS)))
for i, (h, s) in enumerate(REGIONS, 1):
    print("  %2d  %-58s %5d nt" % (i, h.replace(" reverse complement", "")[:58], len(s)))

30 FASTA records -> 21 unique regions after revcomp-aware dedup
   1  SsHADV-1|GQ365709.1|IR1|divergent|AC1                        133 nt
   2  FgGMTV1|NC_078279.1|IR0|terminal|CP                          141 nt
   3  FgGMTV1|NC_078278.1|IR0|terminal|                            205 nt
   4  FgGMTV1|NC_078277.1|IR0|terminal|                             53 nt
   5  BcssDV1|OR146577.1|IR0|tandem_forward|hypothetical protein   894 nt
   6  BcssDV1|OR146576.1|IR0|terminal|hypothetical protein         216 nt
   7  BcssDV1|OR146575.1|IR0|terminal|hypothetical protein         135 nt
   8  BcssDV1|OR146574.1|IR0|tandem_forward|hypothetical protein   895 nt
   9  BcssDV1|OR146572.1|IR0|terminal|hypothetical protein         216 nt
  10  BcssDV1|OR146570.1|IR0|tandem_forward|hypothetical protein   974 nt
  11  BcssDV1|OR146569.1|IR0|terminal|hypothetical protein         216 nt
  12  BcssDV1|OR146568.1|IR0|tandem_forward|hypothetical protein   898 nt
  13  BcssDV1|OR146567.1|IR0|terminal|hypothetic

---
## 4. Self-test — halts the notebook on failure

This is the only evidence that the recipe in §3 is the one that produced your existing CSV. It runs
the pipeline in `best_strand` mode over the same 21 regions and demands **byte-for-byte** equality
with all 21 published chimeras.

It fails if: the Biopython version reorders co-optimal alignments, a scoring parameter is edited, the
graft rule is changed, or the ambiguity handling is changed. Any of those would silently produce a
plausible-looking but different sequence set — which is worse than an error.

In [6]:
_ok = _fail = 0
def _ck(label, cond, extra=""):
    global _ok, _fail
    if cond:
        _ok += 1;  print("  PASS %-52s %s" % (label, extra))
    else:
        _fail += 1; print("  FAIL %-52s %s" % (label, extra))

_al = make_aligner()
_pub_names = list(PUBLISHED)

_ck("30 records parsed", len(RECORDS) == 30, str(len(RECORDS)))
_ck("dedup gives 21 regions", len(REGIONS) == 21, str(len(REGIONS)))
_acc = [h.split("|")[1] for h, _ in REGIONS]
_ck("region order matches published chimera order",
    all(a in n for a, n in zip(_acc, _pub_names)),
    "%d accessions" % len(_acc))

_repro = 0
for _i, (_h, _s) in enumerate(REGIONS):
    _r = build_chimera(DZIKI, _s, _al, "best_strand")
    _repro += (_r["sequence"] == PUBLISHED[_pub_names[_i]])
_ck("best_strand reproduces all 21 published chimeras",
    _repro == 21, "%d/21 byte-identical" % _repro)

# the ambiguity rule, isolated
_amb = [s for _, s in REGIONS if set(s) - set("ACGT")]
_ck("one donor carries IUPAC ambiguity codes", len(_amb) == 1,
    "%d ambiguous bases" % (len(set(_amb[0])) and sum(1 for c in _amb[0] if c not in "ACGT")))

_ck("graft never changes length",
    all(len(build_chimera(DZIKI, s, _al, "as_given")["sequence"]) == 800
        for _, s in REGIONS[:5]))

print("\n" + "=" * 70)
print("SELF-TEST  PASSED %d  FAILED %d" % (_ok, _fail))
print("=" * 70)
assert _fail == 0, ("Self-test failed - the recipe no longer reproduces the known-good "
                    "chimeras. Do NOT submit anything generated below.")

  PASS 30 records parsed                                    30
  PASS dedup gives 21 regions                               21
  PASS region order matches published chimera order         21 accessions


  PASS best_strand reproduces all 21 published chimeras     21/21 byte-identical
  PASS one donor carries IUPAC ambiguity codes              24 ambiguous bases
  PASS graft never changes length                           

SELF-TEST  PASSED 6  FAILED 0


---
## 5. Build the reverse-complement chimera set

`MODE = "as_given"`: each donor is grafted in the orientation the reverse-complement file supplies.
No strand search, so the eight regions whose winning orientation was already the reverse complement
will reproduce chimeras you already have. They are flagged rather than dropped, so the numbering
stays 1:1 with the original 21.

In [7]:
aligner = make_aligner()
_by_seq = {v: k for k, v in PUBLISHED.items()}

CHIMERAS = OrderedDict()
DIAG = []

for i, (header, donor) in enumerate(REGIONS, 1):
    clean = header.replace(" reverse complement", "")
    parts = clean.split("|")
    virus, acc = parts[0], parts[1]

    r = build_chimera(DZIKI, donor, aligner, MODE)
    name = "rc%02d_%s_%s_rc_%dbp_%dchg" % (i, virus, acc, r["n_aligned"], r["n_changed"])

    dup = _by_seq.get(r["sequence"])
    CHIMERAS[name] = r["sequence"]
    DIAG.append(dict(name=name, region_index=i, virus=virus, accession=acc,
                     donor_header=clean, donor_len=r["donor_len"],
                     sw_score=r["score"], n_optimal_alignments=r["n_optimal"],
                     aligned_bp=r["n_aligned"], n_changed=r["n_changed"],
                     ambiguous_skipped=r["n_ambiguous_skipped"],
                     backbone_start=r["target_start"], backbone_end=r["target_end"],
                     duplicate_of=dup or ""))

NEW = OrderedDict((d["name"], CHIMERAS[d["name"]]) for d in DIAG if not d["duplicate_of"])

print("%-42s %6s %6s %6s %6s %11s  %s"
      % ("name", "score", "alnbp", "chg", "ambig", "backbone", "status"))
print("-" * 108)
for d in DIAG:
    print("%-42s %6.0f %6d %6d %6d %5d-%-5d  %s"
          % (d["name"][:42], d["sw_score"], d["aligned_bp"], d["n_changed"],
             d["ambiguous_skipped"], d["backbone_start"], d["backbone_end"],
             ("dup of " + d["duplicate_of"][:34]) if d["duplicate_of"] else "NEW"))

print("\n%d chimeras built  ->  %d NEW, %d already in your existing set"
      % (len(CHIMERAS), len(NEW), len(CHIMERAS) - len(NEW)))

name                                        score  alnbp    chg  ambig    backbone  status
------------------------------------------------------------------------------------------------------------
rc01_SsHADV-1_GQ365709.1_rc_89bp_37chg         34     89     37      0   273-366    NEW
rc02_FgGMTV1_NC_078279.1_rc_132bp_62chg        49    132     62      0   608-742    NEW
rc03_FgGMTV1_NC_078278.1_rc_124bp_56chg        58    124     56      0    95-221    dup of chimera03_FgGMTV1_NC_078278.1_-_12
rc04_FgGMTV1_NC_078277.1_rc_36bp_11chg         29     36     11      0   632-670    dup of chimera04_FgGMTV1_NC_078277.1_-_36
rc05_BcssDV1_OR146577.1_rc_727bp_356chg       123    727    356      0    33-797    NEW
rc06_BcssDV1_OR146576.1_rc_198bp_92chg         57    198     92      0     1-215    NEW
rc07_BcssDV1_OR146575.1_rc_112bp_52chg         42    112     52      0     8-123    NEW
rc08_BcssDV1_OR146574.1_rc_737bp_374chg       120    737    374      0     8-788    NEW
rc09_BcssDV1_OR14657

---
## 6. QC

Every one of these is a hard gate on a property the Judge submission depends on. `N` is asserted to
be zero because the CSV schema carries an `N` column and the endpoint expects pure ACGT.

In [8]:
_ok = _fail = 0

_ck("all sequences exactly 800 nt",
    all(len(s) == 800 for s in CHIMERAS.values()))
_ck("all sequences pure ACGT (no N, no IUPAC)",
    all(set(s) <= set("ACGT") for s in CHIMERAS.values()))
_ck("no duplicates within the new set",
    len(set(CHIMERAS.values())) == len(CHIMERAS),
    "%d distinct / %d" % (len(set(CHIMERAS.values())), len(CHIMERAS)))
_ck("every sequence differs from the backbone",
    all(s != DZIKI for s in CHIMERAS.values()))
_ck("names are unique", len(set(CHIMERAS)) == len(CHIMERAS))

# a chimera must differ from the backbone ONLY inside its aligned block
_span_ok = True
for d in DIAG:
    s = CHIMERAS[d["name"]]
    outside = [j for j in range(800)
               if s[j] != DZIKI[j] and not (d["backbone_start"] <= j < d["backbone_end"])]
    if outside:
        _span_ok = False
        print("    %s: %d edits outside the aligned block" % (d["name"], len(outside)))
_ck("edits confined to the aligned block", _span_ok)

_ck("changed-base count in each name matches the sequence",
    all(sum(1 for x, y in zip(CHIMERAS[d["name"]], DZIKI) if x != y) == d["n_changed"]
        for d in DIAG))

_hd = sorted(sum(1 for x, y in zip(s, DZIKI) if x != y) for s in CHIMERAS.values())
print("\nedit distance from backbone: min %d, median %d, max %d"
      % (_hd[0], _hd[len(_hd) // 2], _hd[-1]))
print("GC%%: backbone %.1f, chimeras %.1f-%.1f"
      % (100 * sum(c in "GC" for c in DZIKI) / 800,
         100 * min(sum(c in "GC" for c in s) for s in CHIMERAS.values()) / 800,
         100 * max(sum(c in "GC" for c in s) for s in CHIMERAS.values()) / 800))

print("\n" + "=" * 70)
print("QC  PASSED %d  FAILED %d" % (_ok, _fail))
print("=" * 70)
assert _fail == 0, "QC failed - do not submit."

  PASS all sequences exactly 800 nt                         
  PASS all sequences pure ACGT (no N, no IUPAC)             
  PASS no duplicates within the new set                     21 distinct / 21
  PASS every sequence differs from the backbone             
  PASS names are unique                                     
  PASS edits confined to the aligned block                  
  PASS changed-base count in each name matches the sequence 

edit distance from backbone: min 4, median 62, max 374
GC%: backbone 47.5, chimeras 44.2-51.9

QC  PASSED 7  FAILED 0


---
## 7. Export

Four files.

- **`pks1_mycovirus_chimeras_revcomp.csv`** — all 21, schema byte-compatible with
  `pks1_mycovirus_chimeras_hackathon.csv` (same 7 columns, `;` delimiter, CRLF). Drop-in.
- **`..._NEW_ONLY.csv`** — the 13 that are not already in your set. This is the one to concatenate
  onto your existing CSV.
- **`..._diagnostics.csv`** — alignment score, co-optimal count, aligned span, ambiguity skips,
  duplicate mapping. Not for submission; for deciding what to submit.
- **`.fasta`** — the 13 new ones, for `POST /wgraj`.

In [9]:
import csv

HEADER = ["nazwa", "gatunek", "gatunek_krotko", "genom", "dlugosc", "N", "sekwencja"]

def write_submission_csv(path, mapping):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh, delimiter=";")
        w.writerow(HEADER)
        for name, seq in mapping.items():
            w.writerow([name, GATUNEK, GATUNEK_KROTKO, GENOM,
                        len(seq), seq.count("N"), seq])
    print("wrote %2d rows -> %s" % (len(mapping), path))

write_submission_csv(OUT_CSV, CHIMERAS)
write_submission_csv(OUT_CSV_NEW, NEW)

with open(OUT_DIAG, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, delimiter=";", fieldnames=list(DIAG[0]))
    w.writeheader()
    w.writerows(DIAG)
print("wrote %2d rows -> %s" % (len(DIAG), OUT_DIAG))

with open(OUT_FASTA, "w") as fh:
    for name, seq in NEW.items():
        fh.write(">%s\n" % name)
        for i in range(0, len(seq), 70):
            fh.write(seq[i:i + 70] + "\n")
print("wrote %2d records -> %s" % (len(NEW), OUT_FASTA))

wrote 21 rows -> pks1_mycovirus_chimeras_revcomp.csv
wrote 13 rows -> pks1_mycovirus_chimeras_revcomp_NEW_ONLY.csv
wrote 21 rows -> pks1_mycovirus_chimeras_revcomp_diagnostics.csv
wrote 13 records -> pks1_mycovirus_chimeras_revcomp.fasta


---
## 8. Feeding these to the Judge

Drop-in for `Score_Chimeras_Judge.ipynb` §2 — replace `CHIMERAS` there with the union. 34 competitors
plus wild-type is 595 round-robin pairs, ~60 s at 600/min, ~120 s if symmetrisation engages.

```python
import csv
def load(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return {r["nazwa"]: r["sekwencja"] for r in csv.DictReader(fh, delimiter=";")}

CHIMERAS = {}
CHIMERAS.update(load("pks1_mycovirus_chimeras_hackathon.csv"))        # your 21
CHIMERAS.update(load("pks1_mycovirus_chimeras_revcomp_NEW_ONLY.csv")) # +13
assert len(CHIMERAS) == 34 and all(len(s) == 800 for s in CHIMERAS.values())
```

If API budget is tight, the §8 head-to-head-against-wild-type screen costs 13 calls and tells you
which of the new ones are worth putting in a round-robin at all.

---

## 9. Limitations — the ones that could actually cost you points

1. **`alignments[0]` is a tie-break, not a result.** Ten of the 21 donors have >1000 co-optimal
   alignments. Different co-optimal paths graft different sequences, all equally justified by the
   score. The set produced here is *one* valid sample from a large equivalence class. Verified
   against Biopython 1.88; §4 fails loudly on any version that reorders them.

2. **The scoring parameters were reverse-engineered, not specified.** `(+2, −1, −5, −2)` was
   recovered by grid search as the unique set reproducing all 21 published chimeras. It is almost
   certainly what was used, but it is an inference from output, not a stated input. Nothing
   downstream depends on it being biologically optimal — it is just the rule that keeps this set
   consistent with your existing one.

3. **Local alignment between unrelated sequences is largely noise.** These are ssDNA mycovirus
   intergenic regions against a fungal PKS1 promoter; there is no homology. A Smith-Waterman hit
   scoring 26–161 over 19–770 bp is mostly chance similarity in AT-rich sequence. The graft is a
   *sequence-shuffling operator*, not a biological alignment, and the result should be judged
   empirically by `/sedzia`, not defended as homologous.

4. **The large grafts destroy the backbone.** `rc05`, `rc08`, `rc10`, `rc12`, `rc14` change 350–375
   of 800 positions. Any TATA-like element, TSS, or upstream regulatory motif in the wild-type is
   gone. Historically these are as likely to be the weakest candidates as the strongest — the
   19–120 bp grafts are the ones preserving backbone architecture.

5. **`rc10` runs to position 800.** Its aligned block ends at the last base, so the graft reaches the
   3' end of the backbone, immediately upstream of where the coding sequence would begin. If the
   Judge is sensitive to core-promoter/TSS proximity, that one is disproportionately likely to score
   badly for a reason unrelated to the donor sequence itself.

6. **Ambiguity handling is a silent no-op, by design.** 24 positions in `OR146568.1` are left as
   backbone bases rather than grafted. `rc12` is therefore 24 positions less "donated" than its name
   suggests. `ambiguous_skipped` in the diagnostics CSV records this per chimera.

7. **Eight duplicates are excluded, not merged.** They carry no new information for the Judge and
   would waste round-robin pairs. If your submission is scored on count rather than quality, revisit
   that choice — `OUT_CSV` retains all 21.